# FEATURE SELECTION & MODEL CREATION

### 1. Import Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2


In [2]:
# Import model functions from Models.py
from Models import (random_forest_grid,dt_grid,nb_model,logistic_grid,svm_grid,cm_prediction)

### 2. Load Pre-processed Data

In [3]:
dataset = pd.read_csv("preprocessed_data.csv")
print(dataset.head())

   age   bmi  alcohol_consumption  sun_exposure  income_level  \
0   79  24.8                    2             0             0   
1   77  39.9                    1             1             1   
2   24  26.4                    0             1             0   
3   69  23.1                    0             0             1   
4   63  29.6                    2             2             0   

   latitude_region  vitamin_a_percent_rda  vitamin_c_percent_rda  \
0                2                  119.1                  147.3   
1                1                   85.7                   57.5   
2                0                   48.3                  152.1   
3                1                   75.8                   51.0   
4                1                   93.3                  111.5   

   vitamin_d_percent_rda  vitamin_e_percent_rda  ...  disease_diagnosis  \
0                 152.88                   97.5  ...                  1   
1                  32.76                   82.7  .

### 3. Split Features & Target

In [4]:
X = dataset.drop('disease_diagnosis', axis=1)
y = dataset['disease_diagnosis']

### 4. Train-Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

### 5. Feature Selection (SelectKBest)

In [6]:
# Select top 10 most important features using Chi-Square test
selector = SelectKBest(score_func=chi2, k=10)

X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

### 6. Feature Scaling

In [7]:
#  Scaling

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


### 7. Model Training & Evaluation

In [8]:
# Run All Models

results = []

In [9]:
# Random Forest
acc, f1, params = random_forest_grid(X_train, y_train, X_test, y_test)
results.append(["Random Forest", acc, f1, params])

# Decision Tree
acc, f1, params = dt_grid(X_train, y_train, X_test, y_test)
results.append(["Decision Tree", acc, f1, params])

# Logistic Regression
acc, f1, params = logistic_grid(X_train, y_train, X_test, y_test)
results.append(["Logistic Regression", acc, f1, params])

# SVM (RBF)
acc, f1, params = svm_grid(X_train, y_train, X_test, y_test)
results.append(["SVM (RBF)", acc, f1, params])

# Naive Bayes
acc, f1 = nb_model(X_train, y_train, X_test, y_test)
results.append(["Naive Bayes", acc, f1, "No Params"])

-----RANDOM FOREST-----
Best Params: {'criterion': 'entropy', 'max_depth': None, 'n_estimators': 100}
Accuracy: 0.935
F1 Score: 0.9358961919731602
Classification _Report:               precision    recall  f1-score   support

           0       0.93      0.92      0.93       297
           1       0.99      0.96      0.98       403
           2       0.97      0.89      0.93        37
           3       0.84      0.91      0.87       238
           4       1.00      1.00      1.00        25

    accuracy                           0.94      1000
   macro avg       0.95      0.94      0.94      1000
weighted avg       0.94      0.94      0.94      1000

-----DECISION TREE-----
Best Params: {'criterion': 'gini', 'max_depth': 10}
Accuracy: 0.924
F1 Score: 0.9235306454871951
Classification _Report:               precision    recall  f1-score   support

           0       0.91      0.94      0.92       297
           1       0.97      0.97      0.97       403
           2       0.94      0.8

### 8. Model Comparison Table

In [10]:
# Convert to DataFrame

results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "F1 Score", "Best Params"])

In [11]:
print("\nFinal Model Comparison:\n")
print(results_df)


Final Model Comparison:

                 Model  Accuracy  F1 Score  \
0        Random Forest     0.935  0.935896   
1        Decision Tree     0.924  0.923531   
2  Logistic Regression     0.823  0.822032   
3            SVM (RBF)     0.862  0.861335   
4          Naive Bayes     0.717  0.711718   

                                         Best Params  
0  {'criterion': 'entropy', 'max_depth': None, 'n...  
1             {'criterion': 'gini', 'max_depth': 10}  
2                       {'C': 10, 'solver': 'lbfgs'}  
3           {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}  
4                                          No Params  


## Load the dataset

In [ ]:
# dataset=pd.read_csv("preprocessed_data.csv")
# dataset

In [ ]:
# dataset["disease_diagnosis"].value_counts()

## Input Output split

In [ ]:
# indep_X = dataset.drop('disease_diagnosis', axis=1)
# dep_Y = dataset['disease_diagnosis']

## Feature Selection

In [ ]:
# 2. Feature Selection Function

def selectkbest(indep_X,dep_Y,n):
        test = SelectKBest(score_func=chi2, k=n)
        fit1= test.fit(indep_X,dep_Y)
        selectk_features = fit1.transform(indep_X)
        return selectk_features

# 3. Data Splitting + Scaling

def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# 4. Prediction + Evaluation Function    
def cm_prediction(classifier, X_test, y_test):
    from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
    y_pred = classifier.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return acc, cm 


# 5. Model Functions

def logistic_grid(X_train, y_train, X_test, y_test):

    param_grid = {
        'solver': ['lbfgs', 'liblinear'],
        'C': [0.1, 1, 10]
    }

    grid = GridSearchCV(
        LogisticRegression(max_iter=1000),
        param_grid,
        scoring='f1_weighted',
        cv=5,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    # Prediction
    y_pred = best_model.predict(X_test)

    # Evaluation
    acc, report, cm = cm_prediction(best_model, X_test, y_test)

    f1 = f1_score(y_test, y_pred, average='weighted')

    print("Best Params:", grid.best_params_)
    print("Accuracy:", acc)
    print("F1 Score:", f1)

    return acc, f1, grid.best_params_

def svm_grid(X_train, y_train, X_test, y_test):

    from sklearn.svm import SVC
    from sklearn.model_selection import GridSearchCV
    from sklearn.metrics import accuracy_score, f1_score

    # Step 1: Parameter grid
    param_grid = {
        'C': [0.1, 1, 10],
        'gamma': [0.1, 0.01],
        'kernel': ['rbf']
    }

    # Step 2: GridSearch
    grid = GridSearchCV(
        SVC(),
        param_grid,
        scoring='f1_weighted',
        cv=5,
        n_jobs=-1
    )

    # Step 3: Train
    grid.fit(X_train, y_train)

    # Step 4: Best model
    best_model = grid.best_estimator_

    # Step 5: Prediction
    y_pred = best_model.predict(X_test)

    # Step 6: Evaluation
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print("Best Params:", grid.best_params_)
    print("Accuracy:", acc)
    print("F1 Score:", f1)
    # Step 7: Best parameters
    best_params = grid.best_params_

    return acc, f1, best_params

def random_forest_grid(X_train, y_train, X_test, y_test):

    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GridSearchCV
    from sklearn.metrics import accuracy_score, f1_score

    # Step 1: Parameter grid
    param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [None, 10, 20],
        'criterion': ['gini', 'entropy']
    }

    # Step 2: GridSearch
    grid = GridSearchCV(
        RandomForestClassifier(random_state=0),
        param_grid,refit=True, verbose=0,
        scoring='f1_weighted',
        cv=5,
        n_jobs=-1
    )

    # Step 3: Train
    grid.fit(X_train, y_train)

    # # Step 4: Best model
    # best_model = grid.best_estimator_

    # Step 5: Prediction
    y_pred = grid.predict(X_test)

    # Step 6: Evaluation
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    report = classification_report(y_test, y_pred)

    # Step 7: Best parameters
    print("Best Params:", grid.best_params_)
    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print("Classification _Report:", report)
    

    return acc, f1, grid.best_params_
    
# def svm_NL(X_train,y_train,X_test):
                
#         from sklearn.svm import SVC
#         classifier = SVC(kernel = 'rbf', random_state = 0)
#         classifier.fit(X_train, y_train)
#         classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
#         return  classifier,Accuracy,report,X_test,y_test,cm
   
# def Navie(X_train,y_train,X_test):       
#         # Fitting K-NN to the Training set
#         from sklearn.naive_bayes import GaussianNB
#         classifier = GaussianNB()
#         classifier.fit(X_train, y_train)
#         classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
#         return  classifier,Accuracy,report,X_test,y_test,cm         
    
    
# def knn(X_train,y_train,X_test):
           
#         # Fitting K-NN to the Training set
#         from sklearn.neighbors import KNeighborsClassifier
#         classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
#         classifier.fit(X_train, y_train)
#         classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
#         return  classifier,Accuracy,report,X_test,y_test,cm
# def Decision(X_train,y_train,X_test):
        
#         # Fitting K-NN to the Training set
#         from sklearn.tree import DecisionTreeClassifier
#         classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
#         classifier.fit(X_train, y_train)
#         classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
#         return  classifier,Accuracy,report,X_test,y_test,cm      


# def random(X_train,y_train,X_test):
        
#         # Fitting K-NN to the Training set
#         from sklearn.ensemble import RandomForestClassifier
#         classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
#         classifier.fit(X_train, y_train)
#         classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
#         return  classifier,Accuracy,report,X_test,y_test,cm

# 6. Result Storage Function

# def selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): 
    
#     dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Logistic','SVMl','SVMnl','KNN','Navie','Decision','Random'])
#     for number,idex in enumerate(dataframe.index):      
#         dataframe['Logistic'][idex]=acclog[number]       
#         dataframe['SVMl'][idex]=accsvml[number]
#         dataframe['SVMnl'][idex]=accsvmnl[number]
#         dataframe['KNN'][idex]=accknn[number]
#         dataframe['Navie'][idex]=accnav[number]
#         dataframe['Decision'][idex]=accdes[number]
#         dataframe['Random'][idex]=accrf[number]
#     return dataframe

In [ ]:
# 7. Loading Dataset
dataset1=pd.read_csv("preprocessed_data.csv", index_col=None)
# df2=dataset1                                     

# 9. Splitting Features & Target
indep_X = dataset1.drop('disease_diagnosis', axis=1)
dep_Y = dataset1['disease_diagnosis']

# kbest=selectkbest(indep_X,dep_Y,10)

In [ ]:
# 9. Train-Test Split FIRST (IMPORTANT)
X_train, X_test, y_train, y_test = train_test_split(
    indep_X, dep_Y, test_size=0.25, random_state=0
)


# 10. Feature Selection ONLY on training data
selector = SelectKBest(score_func=chi2, k=10)

X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)


# 11. Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Run Logistic GridSearch
acc_rf, f1_rf, params_rf = random_forest_grid(X_train, y_train, X_test, y_test)

In [ ]:
# 11. Initialize Accuracy Lists

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [ ]:
# 12. Train-Test Split
X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)  

In [ ]:
# 13. Running All Models

classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test,y_test)
acclog.append(Accuracy)

# classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
# accsvml.append(Accuracy)
    
# classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
# accsvmnl.append(Accuracy)
    
# classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
# accknn.append(Accuracy)
    
# classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
# accnav.append(Accuracy)
    
# classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
# accdes.append(Accuracy)
    
# classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
# accrf.append(Accuracy)

In [ ]:
acc, f1, params = logistic_grid(X_train, y_train, X_test, y_test)


In [ ]:
print("Accuracy:", acc)
print("F1 Score:", f1)
print("Best Params:", params)

In [ ]:
# Result    
# result=selectk_Classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

# result

In [ ]:
results = []

results.append(["ChiSquare", 5, acc, f1])

In [ ]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 1/3, random_state = 0)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {'solver':['newton-cg', 'lbfgs', 'liblinear', 'saga'],
             'penalty':['l2']} 

grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3,n_jobs=1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(X_train, y_train)